## Task 2.5 — SCD Type 2 for Customer Risk Profile

We need to maintain historical tracking of the following customer attributes:

- **risk_rating**
- **kyc_status**
- **account_type**

---

###  Business Requirement

Given a **customer** and a **specific date**, determine:

> What was the customer's risk profile at that point in time?

**Why this matters:**
- Basel III regulatory calculations  
- RBI audit compliance  

---

###  SCD Type 2 — Concept

Instead of updating records in place, we preserve history by **versioning rows**.

> Every change creates a new record while retaining the old one.

---

###  Change Handling Logic

When any tracked attribute changes:

1. **Close the existing record**
   - `is_current = false`
   - `end_date = current_date`

2. **Insert a new record**
   - Updated attribute values
   - `is_current = true`
   - `start_date = current_date`
   - `end_date = NULL`

---

###  Data Structure (Conceptual)

| customer_id | risk_rating | kyc_status | account_type | start_date | end_date | is_current |
|------------|------------|------------|--------------|------------|----------|------------|
| 101        | Low        | Verified   | Savings      | 2023-01-01 | 2023-06-01 | false |
| 101        | Medium     | Verified   | Savings      | 2023-06-01 | NULL       | true  |

---


In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# TASK 2.5 │ SCD Type 2 — Customer Risk Profile History
# Source   │ bfsi.silver_layer.customers_masked
# Target   │ bfsi.silver_layer.dim_customer_risk  (already exists as Delta)
# Rationale│ Basel III requires full audit trail of risk_rating changes to
#           │ accurately reconstruct capital adequacy at any historical date.
# ══════════════════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, DateType
from delta.tables import DeltaTable
from datetime import datetime, date

# ── Catalog / Schema ─────────────────────────────────────────────────────────
CATALOG       = "bfsi"
BRONZE_SCHEMA = "bronze_layer"
SILVER_SCHEMA = "silver_layer"

# ── Source: masked customer table (Task 2.1 output) ──────────────────────────
SOURCE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.customers_masked"

# ── Target: SCD2 dimension (already exists in catalog) ───────────────────────
DIM_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_risk"

# ── Columns whose change triggers a new SCD2 version row ─────────────────────
SCD2_TRACKED_COLS = ["risk_rating", "kyc_status", "account_type"]

# ── Sentinel: open-ended active records use this as effective_end_date ────────
SCD2_OPEN_END_DATE = date(9999, 12, 31)

# ── Job metadata ──────────────────────────────────────────────────────────────
JOB_TS     = datetime.now()
JOB_TS_STR = JOB_TS.isoformat()
TODAY      = date.today()

print("=" * 65)
print(f"  Task 2.5 : SCD Type 2 — Customer Risk Profile")
print(f"  Timestamp: {JOB_TS_STR}")
print(f"  Source   : {SOURCE_TABLE}")
print(f"  Target   : {DIM_TABLE}")
print(f"  Tracking : {SCD2_TRACKED_COLS}")
print("=" * 65)

## Step 1 — Load & Deduplicate Incoming Snapshot

Read `customers_masked`, project only the columns relevant to SCD2,
and deduplicate per `customer_id` keeping the most recently loaded record.
This guards against duplicate rows that can arise from incremental CDC merges
upstream (Task 2.1).

In [0]:
# ── Read source and project only SCD2-relevant columns ───────────────────────
source_df = spark.table(SOURCE_TABLE).select(
    "customer_id",
    "risk_rating",
    "kyc_status",
    "account_type",
    "_load_ts"
)

# ── Deduplicate: keep latest record per customer_id ───────────────────────────
dedup_window = Window.partitionBy("customer_id").orderBy(F.col("_load_ts").desc())

incoming_df = (
    source_df
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

incoming_count = incoming_df.count()
print(f"📥 Incoming records (after dedup): {incoming_count:,}")
incoming_df.show(5, truncate=False)

## Step 2 — Branch: Initial Load vs. Incremental SCD2 Merge

- **Initial load** (target table is empty): bulk-insert all incoming rows directly
  with `_change_reason = 'INITIAL_LOAD'`, `effective_start_date = today`,
  `effective_end_date = 9999-12-31`, `is_current = true`.

- **Incremental** (target has existing data): run the two-step SCD2 MERGE
  described in Step 3.

In [0]:
existing_count = spark.table(DIM_TABLE).count()
IS_INITIAL_LOAD = (existing_count == 0)

print(f"📊 Existing rows in {DIM_TABLE}: {existing_count:,}")
print(f"🔀 Mode: {'INITIAL LOAD' if IS_INITIAL_LOAD else 'INCREMENTAL SCD2 MERGE'}")

## Step 2a — Initial Load Path

In [0]:
if IS_INITIAL_LOAD:
    initial_df = (
        incoming_df
        .withColumn("effective_start_date", F.lit(TODAY).cast(DateType()))
        .withColumn("effective_end_date",   F.lit(SCD2_OPEN_END_DATE).cast(DateType()))
        .withColumn("is_current",           F.lit(True))
        .withColumn("_load_ts",             F.lit(JOB_TS).cast(TimestampType()))
        .withColumn("_change_reason",       F.lit("INITIAL_LOAD"))
        .select(
            "customer_id", "risk_rating", "kyc_status", "account_type",
            "effective_start_date", "effective_end_date",
            "is_current", "_load_ts", "_change_reason"
        )
    )

    (
        initial_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(DIM_TABLE)
    )

    rows_written = initial_df.count()
    print(f"✅ Initial load complete — {rows_written:,} rows written to {DIM_TABLE}")

## Step 2b — Incremental SCD2 MERGE

Two-step atomic pattern on Delta Lake:

| Step | Operation | Condition | Action |
|------|-----------|-----------|--------|
| 1 | `MERGE` UPDATE | `customer_id` matches, `is_current=true`, any tracked col differs | **Expire** old row: set `effective_end_date = today − 1`, `is_current = false` |
| 2 | `MERGE` INSERT | `customer_id` not found with `is_current=true` (new customer, or just expired above) | **Insert** new active row with updated values |

Unchanged customers (all tracked cols identical) are untouched in both steps.

In [0]:
if not IS_INITIAL_LOAD:

    # Register incoming snapshot as temp view for SQL MERGE
    incoming_df.createOrReplaceTempView("incoming_snapshot")

    # Build change-detection predicate from tracked columns
    change_condition = " OR ".join([
        f"target.{col} <> source.{col}" for col in SCD2_TRACKED_COLS
    ])

    # ── MERGE Step 1: Expire stale active records ─────────────────────────────
    spark.sql(f"""
        MERGE INTO {DIM_TABLE} AS target
        USING incoming_snapshot AS source
        ON (
            target.customer_id = source.customer_id
            AND target.is_current = true
        )
        WHEN MATCHED AND ({change_condition}) THEN
            UPDATE SET
                target.effective_end_date = DATE_SUB('{TODAY}', 1),
                target.is_current         = false,
                target._load_ts           = '{JOB_TS_STR}',
                target._change_reason     = 'SCD2_EXPIRED'
    """)
    print("✅ Step 1 complete — expired changed active records")

    # ── Prepare enriched incoming rows for insertion ───────────────────────────
    incoming_enriched = (
        incoming_df
        .withColumn("effective_start_date", F.lit(TODAY).cast(DateType()))
        .withColumn("effective_end_date",   F.lit(SCD2_OPEN_END_DATE).cast(DateType()))
        .withColumn("is_current",           F.lit(True))
        .withColumn("_load_ts",             F.lit(JOB_TS).cast(TimestampType()))
        .withColumn("_change_reason",       F.lit("SCD2_UPDATE"))
        .select(
            "customer_id", "risk_rating", "kyc_status", "account_type",
            "effective_start_date", "effective_end_date",
            "is_current", "_load_ts", "_change_reason"
        )
    )
    incoming_enriched.createOrReplaceTempView("incoming_enriched")

    # ── MERGE Step 2: Insert new versions for changed + brand-new customers ────
    # After Step 1, changed customers no longer have is_current=true,
    # so they fall into the NOT MATCHED branch and get a fresh insert.
    spark.sql(f"""
        MERGE INTO {DIM_TABLE} AS target
        USING incoming_enriched AS source
        ON (
            target.customer_id = source.customer_id
            AND target.is_current = true
        )
        -- Unchanged customers: already active with same values — skip
        WHEN MATCHED THEN
            UPDATE SET target._load_ts = target._load_ts

        -- New customers or just-expired customers: insert fresh active row
        WHEN NOT MATCHED THEN
            INSERT (
                customer_id, risk_rating, kyc_status, account_type,
                effective_start_date, effective_end_date,
                is_current, _load_ts, _change_reason
            )
            VALUES (
                source.customer_id, source.risk_rating, source.kyc_status, source.account_type,
                source.effective_start_date, source.effective_end_date,
                source.is_current, source._load_ts, source._change_reason
            )
    """)
    print("✅ Step 2 complete — inserted new/updated active records")

## Step 3 — Post-Merge Integrity Validation

Two assertions that must hold for a valid SCD2 table:

1. **No duplicate active rows** — every `customer_id` must have exactly one `is_current = true` row
2. **No overlapping date ranges** — for any customer with multiple versions,
   `effective_end_date` of version N must be exactly `effective_start_date` of version N+1 minus 1 day

In [0]:
dim_df = spark.table(DIM_TABLE)

total_rows   = dim_df.count()
current_rows = dim_df.filter(F.col("is_current") == True).count()
hist_rows    = total_rows - current_rows

print("─" * 50)
print(f"  Total rows        : {total_rows:,}")
print(f"  Active (current)  : {current_rows:,}")
print(f"  Historical        : {hist_rows:,}")
print("─" * 50)

# ── Check 1: No duplicate active records per customer ─────────────────────────
dup_check = (
    dim_df
    .filter(F.col("is_current") == True)
    .groupBy("customer_id")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)
dup_count = dup_check.count()

if dup_count > 0:
    print(f"\n❌ INTEGRITY FAILURE: {dup_count} customer(s) have >1 active row!")
    dup_check.show(10)
    raise ValueError(f"SCD2 violation: duplicate active records for {dup_count} customers.")
else:
    print("✅ Check 1 passed — no duplicate active records")

# ── Check 2: No overlapping effective date ranges ─────────────────────────────
overlap_check = spark.sql(f"""
    SELECT a.customer_id,
           a.effective_start_date AS v1_start, a.effective_end_date AS v1_end,
           b.effective_start_date AS v2_start, b.effective_end_date AS v2_end
    FROM   {DIM_TABLE} a
    JOIN   {DIM_TABLE} b
      ON   a.customer_id = b.customer_id
     AND   a.effective_start_date < b.effective_start_date
     AND   a.effective_end_date  >= b.effective_start_date
""")
overlap_count = overlap_check.count()

if overlap_count > 0:
    print(f"\n❌ INTEGRITY FAILURE: {overlap_count} overlapping date ranges found!")
    overlap_check.show(10)
    raise ValueError("SCD2 violation: overlapping effective date ranges.")
else:
    print("✅ Check 2 passed — no overlapping date ranges")

## Step 4 — Point-in-Time Query

**Requirement:** *"What was `CUST001`'s `risk_rating` on `2024-06-01`?"*

The SCD2 predicate: `effective_start_date <= query_date <= effective_end_date`  
returns exactly one row — the version active on that specific date.

In [0]:
QUERY_CUSTOMER = "CUST001"
QUERY_DATE     = "2024-06-01"

pit_df = spark.sql(f"""
    SELECT
        customer_id,
        risk_rating,
        kyc_status,
        account_type,
        effective_start_date,
        effective_end_date,
        is_current,
        _change_reason
    FROM {DIM_TABLE}
    WHERE customer_id          = '{QUERY_CUSTOMER}'
      AND effective_start_date <= DATE('{QUERY_DATE}')
      AND effective_end_date   >= DATE('{QUERY_DATE}')
""")

row_count = pit_df.count()
print(f"🔍 Point-in-time query: {QUERY_CUSTOMER} as of {QUERY_DATE}")
print(f"   Rows returned: {row_count}  (expected: 1)\n")

if row_count == 0:
    print("⚠️  No record found — customer may not exist or date is outside loaded range.")
elif row_count > 1:
    print("❌ ERROR: >1 row returned — SCD2 integrity issue!")
    pit_df.show(truncate=False)
else:
    pit_df.show(truncate=False)
    risk_val = pit_df.collect()[0]["risk_rating"]
    print(f"✅ {QUERY_CUSTOMER}'s risk_rating on {QUERY_DATE} → [{risk_val}]")

## Step 5 — Two-Snapshot Regression Test

Validates the MERGE logic end-to-end using an isolated test table.

- **Snapshot 1** (`2024-01-15`): `CUST001 → risk_rating = LOW`
- **Snapshot 2** (`2024-06-10`): `CUST001 → risk_rating = MEDIUM`

**Expected outcome after Snapshot 2:**
- OLD row: `LOW`, `effective_end_date = 2024-06-09`, `is_current = false`
- NEW row: `MEDIUM`, `effective_end_date = 9999-12-31`, `is_current = true`
- `CUST002` (unchanged): no new row inserted

In [0]:
TEST_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_risk_test_2_5"
spark.sql(f"DROP TABLE IF EXISTS {TEST_TABLE}")

# Create isolated test table with same schema as dim_customer_risk
spark.sql(f"""
    CREATE TABLE {TEST_TABLE} (
        customer_id          STRING  NOT NULL,
        risk_rating          STRING,
        kyc_status           STRING,
        account_type         STRING,
        effective_start_date DATE    NOT NULL,
        effective_end_date   DATE    NOT NULL,
        is_current           BOOLEAN NOT NULL,
        _load_ts             TIMESTAMP,
        _change_reason       STRING
    ) USING DELTA
""")

SCHEMA = ["customer_id","risk_rating","kyc_status","account_type",
          "effective_start_date","effective_end_date","is_current","_load_ts","_change_reason"]

# ── Snapshot 1 ────────────────────────────────────────────────────────────────
s1_date = date(2024, 1, 15)
snap1 = spark.createDataFrame([
    ("CUST001","LOW",   "VERIFIED","SAVINGS", s1_date, SCD2_OPEN_END_DATE, True, JOB_TS, "INITIAL_LOAD"),
    ("CUST002","MEDIUM","PENDING", "CURRENT", s1_date, SCD2_OPEN_END_DATE, True, JOB_TS, "INITIAL_LOAD"),
], SCHEMA)
snap1.write.format("delta").mode("append").saveAsTable(TEST_TABLE)
print(f"📸 Snapshot 1 loaded ({s1_date}): CUST001=LOW, CUST002=MEDIUM")

# ── Snapshot 2: CUST001 changes LOW → MEDIUM, CUST002 unchanged ───────────────
s2_date = date(2024, 6, 10)
snap2 = spark.createDataFrame([
    ("CUST001","MEDIUM","VERIFIED","SAVINGS", s2_date, SCD2_OPEN_END_DATE, True, JOB_TS, "SCD2_UPDATE"),
    ("CUST002","MEDIUM","PENDING", "CURRENT", s2_date, SCD2_OPEN_END_DATE, True, JOB_TS, "SCD2_UPDATE"),
], SCHEMA)
snap2.createOrReplaceTempView("test_snap2")

chg = " OR ".join([f"t.{c} <> s.{c}" for c in SCD2_TRACKED_COLS])

# Step 1 — expire
spark.sql(f"""
    MERGE INTO {TEST_TABLE} t USING test_snap2 s
    ON (t.customer_id = s.customer_id AND t.is_current = true)
    WHEN MATCHED AND ({chg}) THEN
        UPDATE SET t.effective_end_date = DATE_SUB('{s2_date}', 1),
                   t.is_current = false, t._change_reason = 'SCD2_EXPIRED'
""")

# Step 2 — insert
snap2_enriched = snap2.select(*SCHEMA)
snap2_enriched.createOrReplaceTempView("test_snap2_enriched")
spark.sql(f"""
    MERGE INTO {TEST_TABLE} t USING test_snap2_enriched s
    ON (t.customer_id = s.customer_id AND t.is_current = true)
    WHEN MATCHED THEN UPDATE SET t._load_ts = t._load_ts
    WHEN NOT MATCHED THEN INSERT *
""")
print(f"📸 Snapshot 2 applied ({s2_date}): CUST001 → MEDIUM")

# ── Results ───────────────────────────────────────────────────────────────────
result_df = spark.sql(f"""
    SELECT customer_id, risk_rating, effective_start_date,
           effective_end_date, is_current, _change_reason
    FROM {TEST_TABLE} ORDER BY customer_id, effective_start_date
""")
result_df.show(truncate=False)

# ── Assertions ────────────────────────────────────────────────────────────────
rows = {(r.customer_id, r.effective_start_date): r for r in result_df.collect()}

assert rows[("CUST001", s1_date)].is_current == False,                   "FAIL: old row should be closed"
assert rows[("CUST001", s1_date)].effective_end_date == date(2024,6,9),  "FAIL: end date should be 2024-06-09"
assert rows[("CUST001", s2_date)].risk_rating == "MEDIUM",               "FAIL: new row should be MEDIUM"
assert rows[("CUST001", s2_date)].is_current == True,                    "FAIL: new row should be current"
assert ("CUST002", s2_date) not in rows,                                 "FAIL: CUST002 unchanged, no new row expected"
assert rows[("CUST002", s1_date)].is_current == True,                    "FAIL: CUST002 should still be active on snap1 row"

print("✅ All 6 SCD2 assertions passed!")
spark.sql(f"DROP TABLE IF EXISTS {TEST_TABLE}")
print("🧹 Test table cleaned up.")

In [0]:
# ── Collect final metrics for ABC log ────────────────────────────────────────
final_dim     = spark.table(DIM_TABLE)
final_total   = final_dim.count()
final_current = final_dim.filter(F.col("is_current") == True).count()
final_hist    = final_total - final_current

audit_rows = spark.createDataFrame([
    {
        "run_id"        : f"task_2_5_{JOB_TS.strftime('%Y%m%d_%H%M%S')}",
        "check_type"    : "CONTROL",
        "source_system" : "silver_layer.customers_masked",
        "check_name"    : "SCD2_NO_DUPLICATE_ACTIVE_RECORDS",
        "passed"        : True,
        "detail"        : f"All {final_current:,} active records have exactly one is_current=true row per customer_id",
        "check_ts"      : JOB_TS
    },
    {
        "run_id"        : f"task_2_5_{JOB_TS.strftime('%Y%m%d_%H%M%S')}",
        "check_type"    : "CONTROL",
        "source_system" : "silver_layer.customers_masked",
        "check_name"    : "SCD2_NO_OVERLAPPING_DATE_RANGES",
        "passed"        : True,
        "detail"        : f"No overlapping effective_start/end_date ranges found across {final_total:,} total rows",
        "check_ts"      : JOB_TS
    },
    {
        "run_id"        : f"task_2_5_{JOB_TS.strftime('%Y%m%d_%H%M%S')}",
        "check_type"    : "AUDIT",
        "source_system" : "silver_layer.customers_masked",
        "check_name"    : "SCD2_ROW_COUNT_SUMMARY",
        "passed"        : True,
        "detail"        : (
            f"source_rows={incoming_count:,} | "
            f"dim_total={final_total:,} | "
            f"active={final_current:,} | "
            f"historical={final_hist:,} | "
            f"mode={'INITIAL_LOAD' if IS_INITIAL_LOAD else 'INCREMENTAL'}"
        ),
        "check_ts"      : JOB_TS
    }
])

(
    audit_rows.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.abc_audit_log")
)

print("=" * 55)
print("  TASK 2.5 COMPLETE")
print("=" * 55)
print(f"  Source rows       : {incoming_count:,}")
print(f"  Dim total rows    : {final_total:,}")
print(f"  Active records    : {final_current:,}")
print(f"  Historical records: {final_hist:,}")
print(f"  ABC log written   : ✅  (3 rows → abc_audit_log)")
print(f"  Status            : ✅  PASSED")
print("=" * 55)